**Lab type:** debug

**Course:** ML101 — Intro to Machine Learning

**Lesson:** Model Evaluation and Metrics

**Task:** The AI-generated analysis below contains 3 bugs. For each bug: identify what is wrong, explain why the output is misleading, and write the corrected code in the fix cell.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, recall_score, precision_score,
                              roc_auc_score, confusion_matrix)
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n = 2000
# Fraud detection: 2% positive class
fraud = np.zeros(n)
fraud[:40] = 1
np.random.shuffle(fraud)

amount = np.where(fraud == 1, np.random.normal(800, 300, n), np.random.normal(200, 150, n)).clip(10, 3000)
hour = np.where(fraud == 1, np.random.choice(range(0, 6), n), np.random.randint(0, 24, n))
velocity = np.where(fraud == 1, np.random.poisson(8, n), np.random.poisson(2, n))

df = pd.DataFrame({'amount': amount, 'hour': hour, 'velocity': velocity, 'fraud': fraud.astype(int)})
X = df[['amount', 'hour', 'velocity']]
y = df['fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_sc, y_train)
print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {y.mean():.1%}")
print("Setup complete.")

## Step 1: Reporting Model Performance

After training on our imbalanced fraud dataset, the AI analyst reports the model's performance using a single headline metric.

In [ ]:
# AI-generated <- contains Bug 1
y_pred = model.predict(X_test_sc)
acc = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {acc:.3f}")  # <- Bug 1
print("Excellent model — accuracy above 95%!")

**Bug 1 Investigation:** Run the cell above. The accuracy looks high. But with 98% of transactions being legitimate, what accuracy would a model that *always predicts non-fraud* achieve? Is accuracy the right metric for fraud detection?

In [ ]:
# Fix Bug 1 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 2: ROC AUC Score

The analyst now computes the ROC AUC score to get a better sense of the model's ability to separate fraud from non-fraud transactions.

In [ ]:
# AI-generated <- contains Bug 2
roc_auc = roc_auc_score(y_test, y_pred)  # <- Bug 2: y_pred is 0/1, not probabilities
print(f"ROC AUC Score: {roc_auc:.3f}")
print("AUC measures the model's ability to separate classes.")

**Bug 2 Investigation:** Run the cell above. The AUC value is computed, but `y_pred` contains hard 0/1 labels — not probabilities. What does `roc_auc_score` actually need to build the full ROC curve? What method on the model gives you that?

In [ ]:
# Fix Bug 2 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 3: Cross-Validation

To get a more robust estimate of model performance, the analyst uses cross-validation with F1 scoring.

In [ ]:
# AI-generated <- contains Bug 3
cv_scores = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='f1')  # <- Bug 3: cv=5 not stratified
print(f"CV F1 scores: {cv_scores}")
print(f"Mean CV F1: {cv_scores.mean():.3f}")

**Bug 3 Investigation:** Run the cell above. You may see very noisy scores, zeros, or even warnings about undefined F1. With only 2% fraud in the dataset, what happens if a fold ends up with no fraud samples at all? What CV strategy preserves class proportions in every fold?

In [ ]:
# Fix Bug 3 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Corrected Analysis

The cell below applies all three fixes. Run it end-to-end to confirm the evaluation pipeline is now sound.

In [ ]:
# --- Corrected pipeline: all three bugs fixed ---

# Fix 1: report recall, precision, and F1 — not accuracy — for imbalanced fraud detection
y_pred_fixed = model.predict(X_test_sc)
print("=== Classification Metrics ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_fixed):.3f}  (misleading on imbalanced data)")
print(f"Recall:    {recall_score(y_test, y_pred_fixed):.3f}  (fraction of real fraud caught)")
print(f"Precision: {precision_score(y_test, y_pred_fixed):.3f}  (fraction of flagged txns that are fraud)")
print(f"F1 Score:  {f1_score(y_test, y_pred_fixed):.3f}  (harmonic mean of recall and precision)")
print("\nConfusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred_fixed))

# Fix 2: pass predicted probabilities to roc_auc_score, not hard labels
y_proba = model.predict_proba(X_test_sc)[:, 1]
roc_auc_fixed = roc_auc_score(y_test, y_proba)
print(f"\nROC AUC (with probabilities): {roc_auc_fixed:.3f}")

# Fix 3: use StratifiedKFold to preserve class balance in every fold
skf = StratifiedKFold(n_splits=5)
cv_scores_fixed = cross_val_score(model, X_train_sc, y_train, cv=skf, scoring='f1')
print(f"\nStratified CV F1 scores: {cv_scores_fixed}")
print(f"Mean CV F1: {cv_scores_fixed.mean():.3f} (+/- {cv_scores_fixed.std():.3f})")